# Phase 5 — Robustness & Motif Analysis
**Steps 5.1 – 5.6** | Attack simulation, Superpower Fragility Index, frustrated triangle motifs, C(k) hierarchy.

**Deliverable D3:** Fragility Index table + attack/removal curves.

In [ ]:
import os, pickle, warnings
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
from tqdm import tqdm
import community as community_louvain
warnings.filterwarnings('ignore')

matplotlib_style = {'figure.facecolor': '#0d1117', 'axes.facecolor': '#161b22',
                    'axes.edgecolor': '#30363d', 'text.color': 'white',
                    'axes.labelcolor': 'white', 'xtick.color': 'white',
                    'ytick.color': 'white', 'figure.dpi': 150,
                    'grid.color': '#30363d', 'grid.alpha': 0.5}
plt.rcParams.update(matplotlib_style)

ROOT   = os.path.abspath(os.path.join(os.getcwd(), '..'))
NETS   = os.path.join(ROOT, 'results', 'networks')
PLOTS  = os.path.join(ROOT, 'results', 'plots')
TABLES = os.path.join(ROOT, 'results', 'tables')

def load_graph(name):
    with open(os.path.join(NETS, f'{name}.pkl'), 'rb') as f:
        return pickle.load(f)

ERAS = ['full', 'cold_war', 'post_cw', 'post_9_11', 'recent']
G_full = load_graph('full')
print(f'Full network: {G_full.number_of_nodes()} nodes, {G_full.number_of_edges()} edges')

## Steps 5.1 + 5.2 – Targeted vs Random Attack Simulation

In [ ]:
def attack_simulation(G, removal_order, stop_frac=0.5):
    """Simulate iterative node removal. Returns list of (frac_removed, giant_frac)."""
    H = G.copy()
    N_orig = H.number_of_nodes()
    max_remove = int(stop_frac * N_orig)
    results = []

    for i, node in enumerate(removal_order):
        if i >= max_remove:
            break
        if node not in H:
            continue
        H.remove_node(node)
        if H.number_of_nodes() == 0:
            results.append({'removed_frac': (i+1)/N_orig, 'giant_frac': 0})
            break
        if H.number_of_edges() == 0:
            results.append({'removed_frac': (i+1)/N_orig, 'giant_frac': 1/H.number_of_nodes()})
        else:
            gcc = max(nx.connected_components(H), key=len)
            results.append({'removed_frac': (i+1)/N_orig, 'giant_frac': len(gcc)/N_orig})
    return pd.DataFrame(results)


def run_robustness(G, n_random_runs=100, stop_frac=0.5, label=''):
    """Run targeted + random attack simulations."""
    # Targeted: sort by betweenness (descending)
    btw = nx.betweenness_centrality(G, weight='weight', normalized=True)
    targeted_order = sorted(btw, key=btw.get, reverse=True)
    targeted_df = attack_simulation(G, targeted_order, stop_frac)

    # Random: average over many runs
    all_runs = []
    nodes = list(G.nodes)
    for _ in range(n_random_runs):
        rand_order = np.random.permutation(nodes).tolist()
        all_runs.append(attack_simulation(G, rand_order, stop_frac))

    # Interpolate to common x grid
    x_grid = np.linspace(0, stop_frac, 100)
    rand_traces = np.zeros((n_random_runs, len(x_grid)))
    for i, run_df in enumerate(all_runs):
        rand_traces[i] = np.interp(x_grid, run_df['removed_frac'], run_df['giant_frac'],
                                   left=1.0, right=0.0)
    rand_mean = rand_traces.mean(axis=0)
    rand_std  = rand_traces.std(axis=0)

    return targeted_df, x_grid, rand_mean, rand_std, targeted_order


print('Running robustness analysis on full network...')
targeted_df, x_grid, rand_mean, rand_std, targeted_order = run_robustness(
    G_full, n_random_runs=50, stop_frac=0.5, label='full')
print('Done.')

In [ ]:
# Plot targeted vs random for full network
fig, ax = plt.subplots(figsize=(10, 6))
fig.suptitle('Network Robustness: Full UNGA Voting Network', color='white', fontsize=13)

ax.plot(targeted_df['removed_frac'], targeted_df['giant_frac'],
        'o-', color='#f85149', lw=2, ms=4, label='Targeted attack (betweenness)')
ax.plot(x_grid, rand_mean, '-', color='#3fb950', lw=2, label='Random failure (mean)')
ax.fill_between(x_grid, rand_mean - rand_std, rand_mean + rand_std,
                alpha=0.25, color='#3fb950', label='±1 SD random')

ax.set_xlabel('Fraction of nodes removed')
ax.set_ylabel('Giant component (fraction of original N)')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 0.5)
ax.set_ylim(0, 1.05)

plt.tight_layout()
plt.savefig(os.path.join(PLOTS, 'p5_robustness_full.png'), bbox_inches='tight', facecolor='#0d1117')
plt.show()

## Step 5.3 – Robustness on 4 Temporal Sub-networks

In [ ]:
temporal_eras = ['cold_war', 'post_cw', 'post_9_11', 'recent']
era_labels = {'cold_war': 'Cold War (1946–91)', 'post_cw': 'Post-CW (1991–01)',
              'post_9_11': 'Post-9/11 (2001–14)', 'recent': 'Recent (2014–15)'}
era_colors = ['#58a6ff', '#f85149', '#3fb950', '#d2a8ff']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Network Robustness Across Eras', color='white', fontsize=14)
axes = axes.flatten()

era_robustness = {}
for idx, era in enumerate(temporal_eras):
    G_era = load_graph(era)
    if G_era.number_of_edges() < 5:
        print(f'{era}: too few edges, skipping')
        continue
    print(f'\nRunning robustness for {era}...')
    t_df, x_g, r_mean, r_std, t_order = run_robustness(G_era, n_random_runs=30, stop_frac=0.5)
    era_robustness[era] = {'targeted': t_df, 'rand_x': x_g,
                           'rand_mean': r_mean, 'targeted_order': t_order}

    ax = axes[idx]
    ax.plot(t_df['removed_frac'], t_df['giant_frac'],
            'o-', color='#f85149', lw=2, ms=3, label='Targeted')
    ax.plot(x_g, r_mean, '-', color='#3fb950', lw=2, label='Random (mean)')
    ax.fill_between(x_g, r_mean-r_std, r_mean+r_std, alpha=0.2, color='#3fb950')
    ax.set_title(era_labels[era], color='white')
    ax.set_xlabel('Fraction removed'); ax.set_ylabel('Giant component')
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
    ax.set_xlim(0, 0.5); ax.set_ylim(0, 1.05)

plt.tight_layout()
plt.savefig(os.path.join(PLOTS, 'p5_robustness_eras.png'), bbox_inches='tight', facecolor='#0d1117')
plt.show()

## Step 5.4 – Superpower Fragility Index (D3)

In [ ]:
def compute_fragility_index(G, top_n=193):
    """ΔS when each country is removed first (single-node knockout)."""
    N_orig = G.number_of_nodes()
    gcc_orig = max(nx.connected_components(G), key=len)
    S_orig = len(gcc_orig) / N_orig

    fragility = []
    for node in tqdm(list(G.nodes), desc='Fragility index'):
        H = G.copy()
        H.remove_node(node)
        if H.number_of_nodes() == 0:
            delta_S = S_orig
        elif H.number_of_edges() == 0:
            delta_S = S_orig - 1/H.number_of_nodes()
        else:
            gcc_new = max(nx.connected_components(H), key=len)
            S_new = len(gcc_new) / N_orig
            delta_S = S_orig - S_new
        fragility.append({'country': node, 'delta_S': delta_S})

    frag_df = pd.DataFrame(fragility).sort_values('delta_S', ascending=False)
    frag_df['rank'] = range(1, len(frag_df)+1)
    return frag_df

print('Computing Superpower Fragility Index...')
frag_df = compute_fragility_index(G_full)
frag_df.to_csv(os.path.join(TABLES, 'p5_fragility_index_D3.csv'), index=False)

print('\nTop 15 — Superpower Fragility Index (D3):')
print(frag_df.head(15).to_string(index=False))

india_frag = frag_df[frag_df['country'] == 'India']
print('\nIndia:', india_frag.to_string(index=False))

In [ ]:
# Plot fragility index
top10 = frag_df.head(10)
name_map = {'United States of America': 'USA', 'South Africa': 'S.Africa'}
top10_names = [name_map.get(c, c) for c in top10['country']]

fig, ax = plt.subplots(figsize=(10, 6))
fig.suptitle('Superpower Fragility Index — Top 10 (D3)', color='white', fontsize=13)
colors = plt.cm.YlOrRd(np.linspace(0.9, 0.3, 10))
bars = ax.barh(top10_names[::-1], top10['delta_S'].values[::-1], color=colors)
ax.set_xlabel('ΔS (drop in giant component when removed first)')
ax.set_title('Diplomatic Structural Indispensability', color='white')

for bar, val in zip(bars, top10['delta_S'].values[::-1]):
    ax.text(bar.get_width() + 0.001, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=8, color='white')

plt.tight_layout()
plt.savefig(os.path.join(PLOTS, 'p5_fragility_index.png'), bbox_inches='tight', facecolor='#0d1117')
plt.show()

## Step 5.5 – Frustrated Triangle Motif Search

In [ ]:
from itertools import combinations

def count_frustrated_triangles(G, theta_agree=0.7, theta_disagree=0.0):
    """
    Frustrated triangle: A-B agree (w>=theta_agree), B-C agree, A-C disagree (w<theta_disagree).
    Note: In a thresholded graph, 'disagree' means the edge doesn't exist.
    We use the pre-threshold agreement matrix logic.
    """
    nodes = list(G.nodes)
    n_frustrated = 0
    # Simplified: triangle with 2 edges present and 1 missing = frustrated
    for a, b, c in combinations(nodes, 3):
        ab = G.has_edge(a, b)
        bc = G.has_edge(b, c)
        ac = G.has_edge(a, c)
        n_edges = ab + bc + ac
        if n_edges == 2:  # Exactly 2 agreements → structurally frustrated
            n_frustrated += 1
    return n_frustrated


def randomize_graph_degree_preserved(G, n_swaps=10):
    """Return a degree-sequence-preserving random graph via edge swaps."""
    H = G.copy()
    n_edges = H.number_of_edges()
    nx.double_edge_swap(H, nswap=n_swaps * n_edges, max_tries=n_swaps * n_edges * 10)
    return H

print('Counting frustrated triangles in full network...')
print('(Using approximate method — counting 2-edge triangles as frustrated)')

# Subsample for speed
gcc = max(nx.connected_components(G_full), key=len)
G_sample = G_full.subgraph(list(gcc)[:50]).copy()  # Use 50-node subgraph for speed
n_frustrated_real = count_frustrated_triangles(G_sample)
print(f'Frustrated triangles (50-node sample): {n_frustrated_real}')

# Compare to 100 randomized networks
n_rand_runs = 30
rand_frustrated = []
for _ in tqdm(range(n_rand_runs), desc='Randomizing'):
    try:
        H = randomize_graph_degree_preserved(G_sample, n_swaps=5)
        rand_frustrated.append(count_frustrated_triangles(H))
    except Exception:
        pass

if rand_frustrated:
    mean_rand = np.mean(rand_frustrated)
    std_rand  = np.std(rand_frustrated)
    z_score   = (n_frustrated_real - mean_rand) / (std_rand + 1e-9)
    print(f'\nFrustrated triangles:')
    print(f'  Real: {n_frustrated_real}')
    print(f'  Random (mean ± sd): {mean_rand:.1f} ± {std_rand:.1f}')
    print(f'  Z-score: {z_score:.3f}')
    if z_score > 1.96:
        print('  → Significantly MORE frustrated triangles than random (p<0.05)')
    elif z_score < -1.96:
        print('  → Significantly FEWER frustrated triangles (more balanced triads)')

    motif_df = pd.DataFrame({
        'metric': ['frustrated_triangles_real', 'rand_mean', 'rand_std', 'z_score'],
        'value': [n_frustrated_real, mean_rand, std_rand, z_score]
    })
    motif_df.to_csv(os.path.join(TABLES, 'p5_frustrated_motifs.csv'), index=False)

## Step 5.6 – C(k) Hierarchical Model Check (Per Era)

In [ ]:
from scipy import stats as scipy_stats

hier_results = []
fig, ax = plt.subplots(figsize=(9, 6))
fig.suptitle('C(k) Hierarchical Check — All Eras (log-log)', color='white', fontsize=13)

colors_map = {'full':'#58a6ff','cold_war':'#f85149','post_cw':'#3fb950',
              'post_9_11':'#d2a8ff','recent':'#ffa657'}

for era in ERAS:
    G = load_graph(era)
    c_dict = nx.clustering(G)
    degs   = np.array([G.degree(n) for n in G.nodes])
    clusts = np.array([c_dict[n] for n in G.nodes])

    # Bin
    df_ck = pd.DataFrame({'k': degs, 'c': clusts})
    ck = df_ck[df_ck['k'] >= 2].groupby('k')['c'].mean()

    log_k = np.log10(ck.index.astype(float).clip(lower=1))
    log_c = np.log10(ck.values.clip(min=1e-6))
    valid  = np.isfinite(log_k) & np.isfinite(log_c)

    slope = np.nan
    if valid.sum() >= 3:
        slope, intercept, r, p, _ = scipy_stats.linregress(log_k[valid], log_c[valid])
        x_fit = np.linspace(log_k[valid].min(), log_k[valid].max(), 50)
        ax.plot(x_fit, intercept + slope*x_fit, '--', color=colors_map[era], lw=1.5)

    ax.scatter(log_k, log_c, s=20, alpha=0.5, color=colors_map[era],
               label=f'{era} (β={slope:.2f})' if not np.isnan(slope) else era)
    hier_results.append({'era': era, 'ck_slope': round(slope, 3) if not np.isnan(slope) else np.nan})

# Reference line slope=-1
x_ref = np.linspace(0.3, 2.2, 50)
ax.plot(x_ref, -x_ref + 0.2, 'w--', lw=1, alpha=0.3, label='slope=−1 (hierarchical)')
ax.set_xlabel('log₁₀(k)'); ax.set_ylabel('log₁₀ C(k)')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(PLOTS, 'p5_ck_hierarchy_eras.png'), bbox_inches='tight', facecolor='#0d1117')
plt.show()

hier_df = pd.DataFrame(hier_results)
hier_df.to_csv(os.path.join(TABLES, 'p5_hierarchy_slopes.csv'), index=False)
print(hier_df.to_string(index=False))

## ✅ Phase 5 Complete
**Deliverable D3:**
- `results/tables/p5_fragility_index_D3.csv` — Superpower Fragility Index
- `results/plots/p5_robustness_full.png` — attack/random curves, full network
- `results/plots/p5_robustness_eras.png` — attack curves per era
- `results/plots/p5_fragility_index.png` — top-10 fragility bar chart
- `results/tables/p5_frustrated_motifs.csv` — Z-score for frustrated triangles
- `results/tables/p5_hierarchy_slopes.csv` — C(k) slopes per era

**→ Proceed to Notebook 06: Dynamics & Synthesis**